<a href="https://colab.research.google.com/github/wesleymqsss/atelie-generativo-brasilia-arquitetura/blob/main/notebooks/01_dataset.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [6]:
import os
import csv
from transformers import BlipProcessor, BlipForConditionalGeneration
from PIL import Image

# 1. Configuração do modelo
proc = BlipProcessor.from_pretrained("Salesforce/blip-image-captioning-base")
blip = BlipForConditionalGeneration.from_pretrained("Salesforce/blip-image-captioning-base").to("cuda")

# 2. Configurações do seu projeto
PASTA_IMAGENS = "/content/atelie-generativo-brasilia-arquitetura/dados/imagens"
TOKEN_ESTILO = "estilo_arquitetura_brasilia"  # Substitua pelo seu token

# Garantir que a pasta existe
os.makedirs(PASTA_IMAGENS, exist_ok=True)

# 3. Função para processar e gerar rascunho
def gerar_rascunho_legenda(nome_imagem):
    caminho_img = os.path.join(PASTA_IMAGENS, nome_imagem)

    # Abrir, redimensionar para o mínimo exigido (ex: 512x512) e converter para RGB
    img = Image.open(caminho_img).convert("RGB")
    if img.size[0] < 512 or img.size[1] < 512:
        # Redimensiona mantendo a proporção ou forçando (o ideal é crop/redimensionamento adequado)
        img = img.resize((512, 512))
        img.save(caminho_img) # Sobrescreve com a versão corrigida

    # Gerar legenda com o BLIP
    inputs = proc(img, return_tensors="pt").to("cuda")
    saida = blip.generate(**inputs, max_new_tokens=40)
    descricao_blip = proc.decode(saida[0], skip_special_tokens=True)

    # Formata com o token de estilo
    legenda_final = f"{TOKEN_ESTILO}, {descricao_blip}"

    # Salva o rascunho em um arquivo .txt com o mesmo nome da imagem
    nome_txt = nome_imagem.rsplit('.', 1)[0] + ".txt"
    with open(os.path.join(PASTA_IMAGENS, nome_txt), "w", encoding="utf-8") as f:
        f.write(legenda_final)

    print(f"Gerado rascunho para {nome_imagem}: {legenda_final}")

#Exemplo de uso para rodar em todas as imagens da pasta:
for arquivo in os.listdir(PASTA_IMAGENS):
    if arquivo.endswith(('.jpg', '.jpeg', '.png', '.jfif')):
        gerar_rascunho_legenda(arquivo)

Loading weights:   0%|          | 0/473 [00:00<?, ?it/s]

Gerado rascunho para id_imagem_009.jpg: estilo_arquitetura_brasilia, a statue of a man and woman sitting on a bench
Gerado rascunho para id_imagem_024.jpg: estilo_arquitetura_brasilia, the radio tower at the top of the hill
Gerado rascunho para id_imagem_015.jpg: estilo_arquitetura_brasilia, a statue of two people holding hands
Gerado rascunho para id_imagem_005.jpg: estilo_arquitetura_brasilia, a sculpture on top of a white wall
Gerado rascunho para id_imagem_036.jfif: estilo_arquitetura_brasilia, the building is white and has a large circular structure
Gerado rascunho para id_imagem_014.jpg: estilo_arquitetura_brasilia, a statue of two people holding hands
Gerado rascunho para id_imagem_010.jpg: estilo_arquitetura_brasilia, a statue of two people with a bird on their head
Gerado rascunho para id_imagem_034.jpg: estilo_arquitetura_brasilia, a bridge with a light on it and cars driving on the road
Gerado rascunho para id_imagem_035.jpg: estilo_arquitetura_brasilia, a bridge over a body

In [7]:
import os

# Caminho para os arquivos gerados
caminho_imagens = "/content/atelie-generativo-brasilia-arquitetura/dados/imagens"
caminho_final = "/content/atelie-generativo-brasilia-arquitetura/dados/legendas.txt"

# Criar o arquivo consolidado
with open(caminho_final, "w", encoding="utf-8") as outfile:
    for arquivo in sorted(os.listdir(caminho_imagens)):
        if arquivo.endswith(".txt"):
            with open(os.path.join(caminho_imagens, arquivo), "r", encoding="utf-8") as infile:
                conteudo = infile.read().strip()
                # Salva no formato: nome_da_imagem.jpg|legenda
                nome_img = arquivo.replace(".txt", ".jpg") # Ajuste se for .png
                outfile.write(f"{nome_img}|{conteudo}\n")

print(f"Arquivo {caminho_final} gerado com sucesso!")

Arquivo /content/atelie-generativo-brasilia-arquitetura/dados/legendas.txt gerado com sucesso!
